In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25})

ROOT = Path.cwd().parents[2] # ../../../
run_dir = max(p for p in (ROOT / "runs").iterdir() if (p / "events.parquet").exists())

ev = pl.read_parquet(run_dir / "events.parquet")
print(run_dir.name, ev.shape)
ev.head()

In [ ]:
rounds = (
    ev.filter(pl.col("event").is_in(["choose", "play_start", "result"]))
    .with_columns(round=pl.col("event").eq("choose").cum_sum().over("agent_id"))
    .pivot(on="event", index=["agent_id", "round"], values="tick", aggregate_function="first")
    .drop_nulls()
    .with_columns(
        wait=pl.col("play_start") - pl.col("choose"),  # дорога до автомата + очередь
        play=pl.col("result") - pl.col("play_start"),
    )
)

results = ev.filter(pl.col("event") == "result").select(
    "agent_id", pl.col("tick").alias("result"), "delta", "capital_after"
)
rounds = rounds.join(results, on=["agent_id", "result"])

agents = sorted(ev["agent_id"].unique())
rounds.head()

In [ ]:
# pyright: reportUnknownMemberType=false

fig, ax = plt.subplots(figsize=(12, 5))
for a in agents:
    r = rounds.filter(pl.col("agent_id") == a)
    ax.plot(r["result"], r["capital_after"], lw=0.2, label=f"agent {a}")

ax.axhline(0, color="k", lw=0.4)
ax.set(xlabel="tick", ylabel="capital")
ax.legend(fontsize=8, ncol=4)

In [ ]:
# pyright: reportUnknownMemberType=false

window = rounds.filter(pl.col("choose") < 100)

fig, ax = plt.subplots(figsize=(13, 4.5))
for i, a in enumerate(agents):
    r = window.filter(pl.col("agent_id") == a)
    ax.broken_barh(list(zip(r["choose"], r["wait"])), (i - 0.35, 0.7), color="lightsteelblue")

    for start, length, delta in zip(r["play_start"], r["play"], r["delta"]):
        color = "seagreen" if delta > 0 else "indianred"
        ax.broken_barh([(start, length)], (i - 0.35, 0.7), color=color)

ax.set_yticks(range(len(agents)), [f"agent {a}" for a in agents])
ax.set(xlabel="tick", title="Wait (light blue) and play (green — win, red — loss)")